VPixx

In [1]:
%load_ext autoreload
%autoreload 2

# load VPixx libraries
from pgl import pglProPixx
from pgl import pglDataPixx
from pgl import pglTrackPixx3
from pgl import pglLabJack
import numpy as np

# Load PGL libraries and start a PGL window
from pgl import pgl, pglDataPixxDigitalIODevice

pgl = pgl()

# close any existing windows
pgl.cleanUp()


(pgl) Warning: pylink not found, pglEyelink class will not be available. Download with: pip install sr-research-pylink
================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 17:06:14) [Clang 19.1.7 ]
(pgl:checkOS) Running on Mac Studio (Mac16,9) with macOS version: 26.5.2
(pgl:checkOS) Apple M4 Max Cores: 14 (10 Performance and 4 Efficiency) Memory: 36 GB
(pgl:checkOS) GPU: Apple M4 Max (Built-In) 32 cores, Metal 4 support
(pgl:checkOS)   DELL U2724DE [Main Display]: 2560 x 1440 (QHD/WQHD - Wide Quad High Definition) (Unknown type) GammaTable size: 1024
(p

Need to make sure that pypixx library is installed. It is now installed on Koret computer, but had to do some shenanigans to get it into the conda environment, I think because it lives in a shared folder:

(pgl) justin@DN0a237c7d pgl % cp -r /Users/Shared/koret_software/miniforge3/envs/gru/lib/python3.13/site-packages/pypixxlib /Users/Shared/koret_software/miniforge3/envs/pgl/lib/python3.12/site-packages/
(pgl) justin@DN0a237c7d pgl % cp -r /Users/Shared/koret_software/miniforge3/envs/gru/lib/python3.13/site-packages/pypixxlib-*.dist-info /Users/Shared/koret_software/miniforge3/envs/pgl/lib/python3.12/site-packages/

I am working off of the demos that are found [here](https://docs.vpixx.com/python/a-comprehensive-trackpixx3-calibration)

You can find the examples by locating the pypixxlib:  pip show -f pypixxlib | grep -i location
On Koret it is here: /Users/Shared/koret_software/miniforge3/envs/pgl/lib/python3.12/site-packages/pypixxlib

Then the examples are in:

/Users/Shared/koret_software/miniforge3/envs/pgl/lib/python3.12/site-packages/pypixxlib/examples

Using the example: [TPxCalibrationTesting.py](/Users/Shared/koret_software/miniforge3/envs/pgl/lib/python3.12/site-packages/pypixxlib/examples/TPxCalibrationTesting.py)



In [2]:
# setup datapixx is digitalIO
digIO = pglDataPixxDigitalIODevice()
digIO.configureDigitalOutputs()

(pglDataPixxDigitalIODevice:openDPx) Opened DataPixx with firmware version: 27
(pglDataPixxDigitalIODevice:openDPx) Current pixel mode: 0
(pglDataPixxDigitalIODevice:openDPx) DAC schedule: 0
Configured: stimulusOn is [1, 0] on DOut channel 8
Configured: stimulusOff is [1, 0] on DOut channel 1


In [ ]:
# Run this to turn on trigger every vsync
#digIO.enableVsyncTrigger()

# run this to stop it
#digIO.stopDPxSchedules()
digIO.configureDigitalOutputs()

In [4]:
labJack = pglLabJack()
labJack.setupDigitalOutput(channel=1)
labJack.setupDigitalOutput(channel=0)

(pglLabJack) Opened T7 LabJack device via USB connection.
             serialNumber: 470040275 ipAddress: 0 port: 0 maxBytesPerMB: 64
(pglDevice) Cleaning up device of type LabJack
(pglLabJack:setupDigitalOutput) FIO1 configured as output, set to LOW
(pglLabJack:setupDigitalOutput) FIO0 configured as output, set to LOW


In [25]:
for iPulse in range(10):
    digIO.digitalOutputPulse()
    #labJack.digitalOutputPulse(channel=1)
    labJack.digitalOutputPulse(channel=0)
    pgl.waitSecs(0.1)

Set both trigger boxes to B (Labeled I think incorrectly "Mac Studio" on desktop) amd B (guest in rack). Low bits come in through the DB25 cable

In [ ]:
digIO.digitalOutputPulse()
digIO.getError()

In [ ]:
# initialize the track pixx device and check status
#trackPixx3 = pglTrackPixx3(pgl)
#trackPixx3.status()

# Initialize dataPixx 
dataPixx = pglDataPixx()
# add to pgl, so that pgl.poll() will return button press events
pgl.devicesAdd(dataPixx)

In [ ]:
# turn on button mapping to digital IO 
# and pixel mode mapping
dataPixx.setupDigitalOutput()
#dataPixx.enableVsyncTrigger()
#dataPixx.stopDPxSchedules()

In [ ]:
dataPixx.configureDigitalOutputs()

In [ ]:
#dataPixx.openDPx()
frameTime = []
flushTime = []
videoSync = []
everyOther = 1
#pgl.close()
for i in range(600):
    start = pgl.getSecs()
    #dataPixx.sendTrigger("stimulusOn")
    #sendOne = pgl.getSecs()
    #dataPixx.sendTrigger("stimulusOff")
    everyOther = 1 if everyOther==0 else 0
    pgl.clearScreen([0, everyOther, 0])

    videoSync.append(pgl.getSecs())
    #dataPixx.dp.DPxWriteRegCache()
    #dataPixx.dp.DPxWriteRegCacheAfterVideoSync()
    videoSync.append(pgl.getSecs())
    flushTime.append(pgl.flush())
    #dataPixx.sendTrigger("stimulusOff")
    #pgl.clearScreen([0, 0, 0])
    #dataPixx.dp.DPxUpdateRegCacheAfterVideoSync()
    #dataPixx.dp.DPxWriteRegCache()
    end = pgl.getSecs()
    #print(f"time: {(end-start)*1000} sendTrigger = {(sendOne-start)*1000} Vsync={(videoSyncOne-sendOne)*1000}")
    #pgl.flush()
    frameTime.append((end-start)*1000)
    #pgl.flush()
    #pgl.flush()
    #pgl.flush()
    #pgl.flush()
    #pgl.flush()
ignoreFrames = 50
frameTime = frameTime[ignoreFrames:]
print(f"totalTime: median: {np.median(frameTime)} max: {np.max(frameTime)}")
videoSync = np.diff(videoSync)*1000
print(videoSync)
print(f"videoSyncTime: median: {np.median(videoSync)} max: {np.max(videoSync)}")

##flushTime = np.diff(flushTime)*1000
#flushTime= flushTime[ignoreFrames:]
#print(flushTime)
#print(f"median flush: {np.median(flushTime)} max: {np.max(flushTime)}")


In [ ]:
from pypixxlib import _libdpx as dp
dp.DPxOpen()
if not self.dp.DPxIsReady():
    self.getError()
    return

# print status
print(f"(pglDataPixx): Opened DataPixx with firmware version: {dp.DPxGetFirmwareRev()}")
print(f"(pglDataPixx): Current pixel mode: {dp.DPxIsDoutPixelMode()}")
print(f"(pglDataPixx): DAC scheudle: {dp.DPxIsDacSchedRunning()}")
 

In [ ]:
signal = [1, 0]
delay = 0
samplingRate = 1000
signalLength = len(signal)
channel = 8
address=int(8e6)

# Ensure the currentAddress is even; if it's odd, increment by 1
if address % 2 != 0:
    address += 1
    
# Shift each bit in the signal to the left by the value of the channel.
# This positions the bit correctly for the digital output channel.
toggledSignal = [(bit << channel) for bit in signal]

# Write the modified signal (toggledSignal) into the VPixx hardware memory
# at the specified address.
dp.DPxWriteRam(address, toggledSignal)

# After configuring all events, commit changes to the register cache of the
# hardware
dp.DPxWriteRegCache()


for iFrame in range(60):
    # set the schedule
    dp.DPxSetDoutSchedule(delay, samplingRate, signalLength, address)
    dp.DPxStartDoutSched()

    #self.dp.DPxSetDoutSchedRate(1,'video')
    #self.dp.DPxSetDoutSchedRate(1000,'hz')

    # and run it
    #dp.DPxWriteRegCache()
    dp.DPxWriteRegCacheAfterVideoSync()        

In [ ]:
#pgl.fullScreen()
color = 0.1
numFrames = 300
for iFrame in range(numFrames):
    pgl.dots(0,0,color=[0,0,color],dotSize=1.5,units='pix')
    pgl.flush()
    pgl.dots(0,0,color=[0,0,0],dotSize=1.5,units='pix')
    pgl.flush()
pgl.dots(0,0,color=[0,0,0],dotSize=1.5,units='pix')
pgl.flush()


In [ ]:
#dataPixx.openDPx()
#dataPixx.closeDPx()
#dataPixx.status()
dataPixx.openDPx()
dataPixx.stopDPxSchedules()
#dataPixx.enableVsyncTrigger()
#dataPixx.openDPx()
dataPixx.enablePixelMode()


In [ ]:
pgl.getFrameRate()

In [ ]:
dataPixx.enableVsyncTrigger()

In [ ]:
import pypixxlib._libdpx as dp
dp.DPxOpen()
#dp.DPxSelectDevice('PROPixx Ctrl')
#dp.DPxStopDoutSched()
#dp.DPxUpdateRegCache()

#base_address = dp.DPxGetDoutBuffBaseAddr()
#buffer_dout = [0xFFFF, 0]
#dp.DPxSetDoutBuff(base_address, 4)
#dp.DPxWriteRam(base_address, buffer_dout)
#dp.DPxSetDoutSched(0, 2, 'video', 0) 

#dp.DPxUpdateRegCache()

#dp.DPxStartDoutSched()
#dp.DPxUpdateRegCache()


In [ ]:
projector.print()

Next, we should be able to calibrate eye image. running calibrateEyeImage should show the subject's eye on the pgl screen in real time.

In [ ]:
# calibrate eye image
trackPixx3.calibrateEyeImage()
#trackPixx3.getError()
#trackPixx3.dp.TPxSetLEDIntensity(8)
#print(trackPixx3.dp.TPxGetLEDIntensity())
#trackPixx3.getCameraImage()
#Image.fromarray(trackPixx3.getCameraImage()).show()
#pgl.poll()

In [ ]:
(calibrationPoints, rawEyePosition) = trackPixx3.calibrateEyePosition(nCalibrationPoints=5,calibrationWidth=5,calibrationHeight=5)

In [ ]:
print(rawEyePosition)


calibrationsCoeff = trackPixx3.dp.TPxGetCalibCoeffs()
coeffXR = calibrationsCoeff[0:9]
coeffYR = calibrationsCoeff[9:18]
coeffXL = calibrationsCoeff[18:27]
coeffYL = calibrationsCoeff[27:36]

if False:
    np.save("/Users/justin/Desktop/calibrationCoeff.npy",calibrationsCoeff )
    np.save("/Users/justin/Desktop/rawEyePosition.npy",rawEyePosition)
    np.save("/Users/justin/Desktop/calibrationPoints.npy",calibrationPoints)

np.any(rawEyePosition[0]==0.0)

In [ ]:
trackPixx3.displayCalibration(calibrationPoints,rawEyePosition)

In [ ]:
calibrationPoints = np.load("/Users/justin/Desktop/calibrationPoints.npy")
calibrationsCoeff = np.load("/Users/justin/Desktop/calibrationCoeff.npy")
rawEyePosition = np.load("/Users/justin/Desktop/rawEyePosition.npy")

In [ ]:
# init proPixx projector and DataPixx
projector = pglProPixx()


In [ ]:
pgl.devicesAdd(datapixx)

In [ ]:
# check status
projector.status()
datapixx.status()

With the datapixx running, then when you do poll, you should get back what buttons were pressed

In [ ]:
# check button presses
datapixx.poll()

If you enableButtonSchedules, it should turn button presses into digital outputs (see code for details)

In [ ]:
# try to enable button schedule - which should convert buttons into digital outputs
#datapixx.enableButtonSchedules()
#datapixx.enablePixelMode()
datapixx.setupDigitalOutput()

In [ ]:
import pypixxlib._libdpx as dp

In [ ]:

#from pypixxlib.propixx import PROPixx
#pgl.verbose=1
#projector = pglProPixx()
#datapixx = pglDataPixx()

for i in range(1000):
    events = pgl.poll()
    #events = datapixx.poll()
    if events is not None:
        for event in events:
            print(event)
    pgl.waitSecs(0.1)
#datapixx.setupDigitalOutput()
#datapixx.test()
#for i in range(5):
#    print(f"Polling DataPixx: {i}")
    #datapixx.poll()
    #pgl.waitSecs(1)


#device.setRearProjection(True)
#from pypixxlib.propixx import PROPixx
#propixx = PROPixx()
#propixx.getDlpSequencerProgram()
#projector.status()
#print(device)
#from pgl.pglDevice import pglDevice, pglProPixx
#huh = pglDevice(pgl,'huh')
#duh = pglProPixx(pgl)